In [ ]:
!pip install -q sentence-transformers scikit-learn pandas numpy joblib


In [ ]:
!pip install -q transformers

In [ ]:
!pip uninstall -y transformers sentence-transformers huggingface-hub
!pip install -q \
    transformers==4.36.2 \
    sentence-transformers==2.2.2 \
    huggingface-hub==0.19.4


Found existing installation: transformers 4.36.2
Uninstalling transformers-4.36.2:
  Successfully uninstalled transformers-4.36.2
Found existing installation: sentence-transformers 2.2.2
Uninstalling sentence-transformers-2.2.2:
  Successfully uninstalled sentence-transformers-2.2.2
Found existing installation: huggingface-hub 0.36.0
Uninstalling huggingface-hub-0.36.0:
  Successfully uninstalled huggingface-hub-0.36.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 311.7/311.7 kB 14.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.0.0 requires huggingface-hub>=0.24.0, but you have huggingface-hub 0.19.4 which is incompatible.
diffusers 0.36.0 requires huggingface-hub<2.0,>=0.34.0, but you have huggingface-hub 0.19.4 which is incompatible.
gradio 5.50.0 requires huggingface-hub<2.0,>=0.33.5, but you have huggingface-hub 0.19.4 which is 

In [ ]:
from sentence_transformers import SentenceTransformer
import transformers
import huggingface_hub

print("Sentence-Transformers OK")
print("Transformers:", transformers.__version__)
print("HF Hub:", huggingface_hub.__version__)


/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


Sentence-Transformers OK
Transformers: 4.36.2
HF Hub: 0.19.4


In [1]:
import pandas as pd
import numpy as np
import joblib
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [ ]:
import pandas as pd

fake_path = "/content/drive/MyDrive/DL-projects/Fake-news-dataset/Fake.csv"
true_path= "/content/drive/MyDrive/DL-projects/Fake-news-dataset/True.csv"


In [ ]:
fake_df = pd.read_csv(fake_path)[["title", "text"]]
fake_df["label"] = 0

real_df = pd.read_csv(true_path)[["title", "text"]]
real_df["label"] = 1

df = pd.concat([fake_df, real_df], axis=0)
df = df.sample(frac=1).reset_index(drop=True)

df["text"] = df["title"] + " " + df["text"]
df = df[["text", "label"]]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df["text"], df["label"], test_size=0.2, random_state=42
)


In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)


Using device: cuda


In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer(
    "all-MiniLM-L6-v2",
    device=device
)


In [ ]:
def truncate_text(text, max_chars=1000):
    return text[:max_chars]

X_train_trunc = X_train.apply(truncate_text)
X_test_trunc = X_test.apply(truncate_text)


In [ ]:
X_train_emb = embedder.encode(
    X_train_trunc.tolist(),
    batch_size=128,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

X_test_emb = embedder.encode(
    X_test_trunc.tolist(),
    batch_size=128,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)


Batches:   0%|          | 0/281 [00:00<?, ?it/s]

Batches:   0%|          | 0/71 [00:00<?, ?it/s]

In [ ]:
embedder = SentenceTransformer("all-MiniLM-L6-v2")

X_train_emb = embedder.encode(
    X_train.tolist(),
    show_progress_bar=True,
    convert_to_numpy=True
)

X_test_emb = embedder.encode(
    X_test.tolist(),
    show_progress_bar=True,
    convert_to_numpy=True
)


Batches:   0%|          | 0/1123 [00:00<?, ?it/s]

Batches:   0%|          | 0/281 [00:00<?, ?it/s]

In [ ]:
model = LogisticRegression(
    max_iter=2000,
    n_jobs=-1,
    class_weight="balanced"
)

model.fit(X_train_emb, y_train)


LogisticRegression(class_weight='balanced', max_iter=2000, n_jobs=-1)

In [ ]:
preds = model.predict(X_test_emb)
print("Accuracy:", accuracy_score(y_test, preds))


Accuracy: 0.9497772828507796


In [ ]:
joblib.dump(model, "fake_news_model.pkl")
joblib.dump(embedder, "sentence_embedder.pkl")


['sentence_embedder.pkl']

In [ ]:
SUSPICIOUS_KEYWORDS = [
    "breaking", "shocking", "secret", "they don't want you to know",
    "miracle", "exposed", "unbelievable", "guaranteed"
]

def heuristic_score(text):
    score = 0
    text_lower = text.lower()
    for kw in SUSPICIOUS_KEYWORDS:
        if kw in text_lower:
            score += 1
    return score


In [2]:
import joblib
from sentence_transformers import SentenceTransformer

# Load saved components
embedder = SentenceTransformer("all-MiniLM-L6-v2")
model = joblib.load("/content/drive/MyDrive/DL-projects/Saved-models/final-fake-news-saved_models/final/fake_news_model.pkl")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [3]:
# model version 1

def predict_fake_news(text):
    # Step 1: Convert text → embedding
    embedding = embedder.encode([text])

    # Step 2: Get probabilities
    probs = model.predict_proba(embedding)[0]
    fake_prob, real_prob = probs

    # Step 3: Decide verdict
    if fake_prob > real_prob:
        verdict = "FAKE"
        confidence = fake_prob
    else:
        verdict = "REAL"
        confidence = real_prob

    # Step 4: Handle uncertainty
    if confidence < 0.65:
        verdict = "UNCERTAIN"

    return {
        "verdict": verdict,
        "confidence": confidence,
        "source": "MiniLM + LogisticRegression",
        "raw_probs": {
            "fake": round(fake_prob, 3),
            "real": round(real_prob, 3)
        }
    }


In [29]:
def generate_explanation(result):
    if result["final_verdict"] == "FAKE (Scientific Impossibility)":
        return "The article contains claims that are scientifically impossible or highly implausible, suggesting strong indicators of misinformation."
    elif result["final_verdict"] == "FAKE (High Confidence)":
        return "The article shows strong patterns commonly associated with misinformation, such as exaggerated claims, unverifiable assertions, or sensational language, and also triggers multiple heuristic red flags."
    elif result["final_verdict"] == "LIKELY FAKE":
        return "The article exhibits several characteristics of fake news, combining model predictions with heuristic indicators like suspicious keywords, questionable claims, or misuse of authority."
    elif result["final_verdict"] == "LIKELY REAL":
        return "While the article might contain some minor questionable elements, the overall analysis from the model and heuristics suggests it is likely real. It maintains a more factual tone and structured writing style, without significant misinformation indicators."
    else:
        return (
            "The system detected mixed signals and insufficient confidence "
            "to make a definitive classification based on the base model alone." # This case should ideally not be reached with final_verdict
        )


def explainable_output(result):
    verdict = result["final_verdict"] # Use the final verdict from the comprehensive system
    # Use the confidence from the base result or calculate a combined confidence if needed
    # For simplicity, we'll keep the base model's confidence for now, but associate it with the final verdict
    confidence = result["confidence"] * 100

    if verdict.startswith("FAKE"): # Check for any 'FAKE' verdict type
        icon = "🚨"
    elif verdict.startswith("LIKELY FAKE"): # Check for any 'LIKELY FAKE' verdict type
        icon = "⚠️"
    elif verdict.startswith("LIKELY REAL"): # Check for any 'LIKELY REAL' verdict type
        icon = "✅"
    else:
        icon = "❓" # For unexpected verdicts

    print(f"{icon} Final Verdict: {verdict}")
    print(f"📈 Base Model Confidence: {confidence:.2f}% (for original verdict: {result['verdict']})")
    print(f"🤖 Model Used: {result['source']}")
    print(f"📝 Explanation: {generate_explanation(result)}")
    print(f"Additional Heuristic Insights:")
    print(f" - Claim Density Score: {result['claim_density']}")
    print(f" - Authority Abuse Score: {result['authority_abuse']}")
    print(f" - Scientific Impossibility Flag: {result['scientific_impossibility']}")

In [30]:
import re

EXTRAORDINARY_TERMS = [
    "quantum", "crisper", "nanobot", "brainwave",
    "hypersonic", "classified", "leaked",
    "99%", "100%", "instant", "erases memory",
    "secret report", "silenced", "buried data"
]

def claim_density_score(text):
    text = text.lower()
    hits = sum(1 for term in EXTRAORDINARY_TERMS if term in text)
    length_factor = max(len(text.split()) / 40, 1)
    return hits / length_factor


In [31]:
IMPOSSIBLE_PATTERNS = [
    r"predict(s|ing)? criminal behavior",
    r"erase(s|d)? memories instantly",
    r"brainwave control",
    r"covid mutation reversal",
    r"quantum ai chip in aadhaar",
    r"5g.*medical",
]

def scientific_impossibility_flag(text):
    text = text.lower()
    for pattern in IMPOSSIBLE_PATTERNS:
        if re.search(pattern, text):
            return True
    return False


In [32]:
SENSITIVE_AUTHORITIES = [
    "isro", "drdo", "rbi", "icmr", "csir",
    "iit", "government", "classified"
]

ABUSE_TERMS = [
    "secret", "leaked", "buried", "silenced",
    "backdoor", "surveillance", "hack"
]

def authority_abuse_score(text):
    t = text.lower()
    score = 0
    for auth in SENSITIVE_AUTHORITIES:
        if auth in t:
            for abuse in ABUSE_TERMS:
                if abuse in t:
                    score += 1
    return score


In [37]:
# model version 2 - optimized

def final_fake_news_system(text):
    base_result = predict_fake_news(text)

    claim_score = claim_density_score(text)
    impossible = scientific_impossibility_flag(text)
    authority_score = authority_abuse_score(text)

    # Access the fake probability from 'raw_probs' dictionary
    risk_score = (
        0.4 * base_result["raw_probs"]["fake"]
        + 0.3 * min(claim_score, 1.5)
        + 0.3 * min(authority_score / 3, 1)
    )

    if impossible:
        verdict = "FAKE (Scientific Impossibility)"
    elif risk_score > 0.7:
        verdict = "FAKE (High Confidence)"
    elif risk_score > 0.5:
        verdict = "LIKELY FAKE"
    else:
        verdict = "LIKELY REAL"

    return {
        **base_result,
        "claim_density": round(claim_score, 2),
        "authority_abuse": authority_score,
        "scientific_impossibility": impossible,
        "final_verdict": verdict
    }



In [42]:
result = final_fake_news_system(
"SHOCKING: CSIR labs confirm cow urine + 5G reverses COVID mutations; 92% efficacy in Phase-3 trials but ICMR buries data to push mRNA boosters—leaked patient logs show full recovery in 72 hours."
)

explainable_output(result)

⚠️ Final Verdict: LIKELY FAKE
📈 Base Model Confidence: 97.31% (for original verdict: REAL)
🤖 Model Used: MiniLM + LogisticRegression
📝 Explanation: The article exhibits several characteristics of fake news, combining model predictions with heuristic indicators like suspicious keywords, questionable claims, or misuse of authority.
Additional Heuristic Insights:
 - Claim Density Score: 1.0
 - Authority Abuse Score: 2
 - Scientific Impossibility Flag: False


In [46]:
joblib.dump(model, "/content/drive/MyDrive/DL-projects/Saved-models/final-fake-news-saved_models/final/final_fake_news_model.pkl")
joblib.dump(embedder, "/content/drive/MyDrive/DL-projects/Saved-models/final-fake-news-saved_models/final/final_sentence_embedder.pkl")


['/content/drive/MyDrive/DL-projects/Saved-models/final-fake-news-saved_models/final/final_sentence_embedder.pkl']